# Chapter 87: Complete Security Case Study — The AI Security Fabric

**Volume 4 — Security Operations | FINAL CHAPTER**

> *"Everything works in isolation. Nothing works together."*
> This chapter fixes that.

You built five specialized security systems across Chapters 70-83:
- **Ch70** — AI-Powered Threat Detection (UEBA, behavioral baselines)
- **Ch72** — SIEM Integration (log normalization, CEP correlation)
- **Ch75** — Network Anomaly Detection (autoencoder, EWMA)
- **Ch80** — Securing AI Systems (prompt injection, output guardrails)
- **Ch83** — Compliance Automation (policy-as-code, drift detection)

Now we wire them into a **unified AI Security Fabric** — a multi-agent architecture
where the whole is greater than the sum of its parts.

**What you will build:**
1. Unified Data Model — the shared schema all agents read from and write to
2. Specialist Worker Agents — four domain experts that analyze in parallel
3. Security Coordinator — synthesizes worker findings into incident briefs
4. Actor-Critic Response Chain — proposes actions, validates safety, auto-executes or escalates
5. Full AI Security Fabric — end-to-end: ingest -> detect -> investigate -> respond -> audit


In [ ]:
# Setup
import os, json, hashlib, re, datetime, uuid, time
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any
from collections import defaultdict

try:
    import anthropic
    _client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", ""))
    _has_llm = bool(os.environ.get("ANTHROPIC_API_KEY"))
except ImportError:
    _client = None
    _has_llm = False

def llm_query(prompt: str, system: str, max_tokens: int = 400,
              model: str = "claude-haiku-4-5-20251001") -> str:
    """Call Claude or return a simulated response based on keywords."""
    if _has_llm and _client:
        msg = _client.messages.create(
            model=model, max_tokens=max_tokens,
            system=system,
            messages=[{"role": "user", "content": prompt}],
        )
        return msg.content[0].text

    # --- Simulation fallback: keyword-driven contextual responses ---
    p = prompt.lower()

    # Threat detection worker simulation
    if "behavioural" in p or "ueba" in p or "authentication" in p:
        if "tor" in p or "185.220" in p or "geo_country" in p:
            return json.dumps({
                "finding": "Impossible travel attack - login from Tor exit node RU, last seen US 1.5h ago",
                "confidence": 0.96,
                "mitre_technique": "T1078 - Valid Accounts",
                "severity": "CRITICAL",
                "indicators": ["Tor exit node", "geo mismatch", "MFA not used", "prod DB target"]
            }, indent=2)
        return json.dumps({"finding": "No anomaly", "confidence": 0.85, "severity": "LOW"}, indent=2)

    # SIEM correlation worker simulation
    if "correlate" in p or "siem" in p or "pattern" in p:
        if "db-server" in p or "prod" in p:
            return json.dumps({
                "pattern_detected": "Data Exfil Precursor - unauthorized access to prod database",
                "related_events": ["VPN auth failure x3", "service account reuse", "large SELECT query"],
                "attack_stage": "Initial Access -> Credential Use -> Collection",
                "confidence": 0.91,
                "mitre_chain": ["T1078", "T1213", "T1005"]
            }, indent=2)
        return json.dumps({"pattern_detected": "None", "confidence": 0.6}, indent=2)

    # Anomaly worker simulation
    if "netflow" in p or "anomaly" in p or "traffic" in p:
        if "185.220" in p or "exfil" in p or "bytes" in p:
            return json.dumps({
                "anomaly": "NetFlow reconstruction error 8.7x above threshold",
                "ewma_spike": "Outbound bytes to untrusted IP: 3.2 GB in 8 min",
                "confidence": 0.89,
                "severity": "HIGH",
                "detail": "Autoencoder MSE=0.87 (threshold=0.10), EWMA trend_score=12.4"
            }, indent=2)
        return json.dumps({"anomaly": "None detected", "confidence": 0.75}, indent=2)

    # Compliance worker simulation
    if "compliance" in p or "pci" in p or "hipaa" in p or "config" in p:
        return json.dumps({
            "drift_detected": True,
            "control_failures": ["PCI-8.3.1 MFA bypass via local fallback", "SOC2-CC7.2 no centralized log"],
            "severity": "HIGH",
            "evidence_logged": True
        }, indent=2)

    # Coordinator synthesis simulation
    if "synthesise" in p or "synthesize" in p or "coordinator" in p or "incident brief" in p:
        return (
            "SEVERITY: CRITICAL | CONFIDENCE: 0.94\n"
            "ATTACK CHAIN: Tor exit node (RU) -> Valid credential use (admin.chen) "
            "-> Prod DB access (db-server-prod-01) -> Data collection (3.2 GB exfil)\n"
            "MITRE: T1078 (Valid Accounts) -> T1213 (Data from Info Repo) -> T1041 (Exfil over C2)\n"
            "AFFECTED ASSETS: db-server-prod-01, admin.chen account, customer PII store\n"
            "ACTION: ISOLATE db-server-prod-01, DISABLE admin.chen, BLOCK 185.220.101.47\n"
            "HUMAN APPROVAL: REQUIRED (CRITICAL severity, production asset impact)"
        )

    # Critic evaluation simulation
    if "critic" in p or "safe" in p or "blast radius" in p or "reversible" in p:
        if "isolate" in p or "disable" in p:
            return json.dumps({
                "verdict": "APPROVED_WITH_CONDITIONS",
                "risk_level": "MEDIUM",
                "reversible": True,
                "blast_radius": "Single host isolation - 2 dependent services identified",
                "conditions": ["Notify on-call team before execution", "Set rollback timer 4h"],
                "confidence_check": "Source confidence 0.94 exceeds 0.85 threshold for auto-action",
                "auto_execute": False,
                "reason": "Production asset - requires human approval per Walk-phase policy"
            }, indent=2)
        return json.dumps({"verdict": "APPROVED", "auto_execute": True, "risk_level": "LOW"}, indent=2)

    return json.dumps({"result": "Analysis complete", "confidence": 0.80}, indent=2)

# ── The suspicious event used throughout the entire case study ────────────────
SUSPICIOUS_EVENT = {
    "id":                 "EVT-2026-0042",
    "timestamp":          "2026-02-26T03:14:22Z",
    "type":               "authentication",
    "user":               "admin.chen",
    "src_ip":             "185.220.101.47",   # known Tor exit node
    "dst_device":         "db-server-prod-01",
    "auth_result":        "success",
    "mfa_used":           False,
    "bytes_transferred":  3_421_000_000,      # 3.2 GB outbound in 8 min
    "geo_country":        "RU",
    "normal_geo":         "US",
    "last_login_hours_ago": 1.5,
    "last_src_ip":        "10.0.1.55",        # normal internal IP
    "dst_port":           5432,               # PostgreSQL
}

print("AI Security Fabric -- Chapter 87 setup complete.")
print(f"LLM mode: {'Claude API' if _has_llm else 'Simulation (set ANTHROPIC_API_KEY to use real LLM)'}")
print(f"\nCase study event: {SUSPICIOUS_EVENT['user']} from {SUSPICIOUS_EVENT['src_ip']}"
      f" ({SUSPICIOUS_EVENT['geo_country']}) -> {SUSPICIOUS_EVENT['dst_device']}")


## Demo 1: Unified Data Model — The Foundation of the Security Fabric

**The Problem:** Five separate systems (Ch70-83) each use their own data format.
The threat detector speaks one schema. The SIEM speaks another. They cannot talk to each other.

**The Solution:** Define three shared schemas upfront. Every component reads from and
writes to these schemas — no exceptions.

```
SecurityEvent    -- any observation from any source (auth, netflow, endpoint, alert)
Investigation    -- coordinator's working state for a group of related events
ResponseAction   -- proposed or executed response with full audit trail
```

> **Network Analogy:** This is the Common Information Model (CIM) principle from Ch72 SIEM,
> now applied to the *entire security stack*. Same idea as having one vendor-neutral
> schema for all your network devices -- it's what makes the fabric possible.

**The SecurityEventStore is the single source of truth.** All agents read from it.
All agents write back to it. The coordinator never needs to ask "which format is this?".


In [ ]:
# Demo 1: Unified Data Model

@dataclass
class SecurityEvent:
    """Normalized representation of any security-relevant observation."""
    event_id:     str
    timestamp:    str
    source_type:  str      # netflow | auth_log | endpoint | siem_alert | compliance
    device:       str
    src_ip:       str
    dst_ip:       str
    user:         str
    action:       str
    confidence:   float    # 0.0 - 1.0
    severity:     str      # LOW | MEDIUM | HIGH | CRITICAL
    raw_data:     Dict     = field(default_factory=dict)
    tags:         List[str] = field(default_factory=list)

@dataclass
class WorkerFinding:
    """A specialist worker's analysis result."""
    worker_name:  str
    event_id:     str
    finding:      str
    confidence:   float
    severity:     str
    mitre_tags:   List[str] = field(default_factory=list)
    raw_output:   str = ""

@dataclass
class Investigation:
    """Coordinator's unified incident record."""
    inv_id:           str
    triggered_by:     str                  # event_id that started the investigation
    events:           List[SecurityEvent]  = field(default_factory=list)
    worker_findings:  List[WorkerFinding]  = field(default_factory=list)
    attack_chain:     List[str]            = field(default_factory=list)
    affected_assets:  List[str]            = field(default_factory=list)
    severity:         str  = "UNKNOWN"
    confidence:       float = 0.0
    incident_brief:   str  = ""
    status:           str  = "OPEN"        # OPEN | ESCALATED | RESOLVED
    created_at:       str  = field(default_factory=lambda: datetime.datetime.now().isoformat())

@dataclass
class ResponseAction:
    """Proposed or executed response action with full audit trail."""
    action_id:        str
    inv_id:           str
    action_type:      str      # ISOLATE_HOST | BLOCK_IP | DISABLE_ACCOUNT | ADD_WATCHLIST | NOTIFY
    target:           str
    parameters:       Dict     = field(default_factory=dict)
    proposed_by:      str  = "actor-agent"
    critic_verdict:   str  = "PENDING"     # APPROVED | REJECTED | APPROVED_WITH_CONDITIONS
    auto_execute:     bool = False
    approved_by:      str  = ""            # "autonomous" or analyst name
    executed_at:      str  = ""
    reversible:       bool = True
    rollback_notes:   str  = ""

class SecurityEventStore:
    """Central store for all security events and investigations."""

    def __init__(self):
        self._events: Dict[str, SecurityEvent] = {}
        self._investigations: Dict[str, Investigation] = {}
        self._audit: List[Dict] = []        # immutable append-only audit trail

    def ingest(self, event: SecurityEvent):
        """Ingest a normalized event."""
        self._events[event.event_id] = event
        self._log("INGEST", event.event_id, f"source={event.source_type} sev={event.severity}")

    def get_event(self, event_id: str) -> Optional[SecurityEvent]:
        return self._events.get(event_id)

    def open_investigation(self, event: SecurityEvent) -> Investigation:
        """Open a new investigation triggered by a high-confidence event."""
        inv = Investigation(
            inv_id       = f"INV-{event.event_id}",
            triggered_by = event.event_id,
            events       = [event],
        )
        self._investigations[inv.inv_id] = inv
        self._log("INVESTIGATE_OPEN", inv.inv_id, f"triggered by {event.event_id}")
        return inv

    def update_investigation(self, inv: Investigation):
        self._investigations[inv.inv_id] = inv
        self._log("INVESTIGATE_UPDATE", inv.inv_id, f"status={inv.status} sev={inv.severity}")

    def _log(self, action: str, obj_id: str, details: str):
        self._audit.append({
            "ts": datetime.datetime.now().isoformat(timespec="seconds"),
            "action": action, "id": obj_id, "details": details
        })

    def print_audit_trail(self, last_n: int = 10):
        print(f"\n  -- Audit Trail (last {last_n}) --")
        for entry in self._audit[-last_n:]:
            print(f"  [{entry['ts']}] {entry['action']:<22} {entry['id']:<25} {entry['details']}")

# ── Ingest the suspicious event into the unified store ─────────────────────────
store = SecurityEventStore()

ev = SecurityEvent(
    event_id    = SUSPICIOUS_EVENT["id"],
    timestamp   = SUSPICIOUS_EVENT["timestamp"],
    source_type = "auth_log",
    device      = SUSPICIOUS_EVENT["dst_device"],
    src_ip      = SUSPICIOUS_EVENT["src_ip"],
    dst_ip      = "10.10.50.5",           # db internal IP
    user        = SUSPICIOUS_EVENT["user"],
    action      = "authentication_success",
    confidence  = 0.95,
    severity    = "CRITICAL",
    raw_data    = SUSPICIOUS_EVENT,
    tags        = ["tor-exit-node", "geo-mismatch", "no-mfa", "prod-db"],
)
store.ingest(ev)

# Open investigation
investigation = store.open_investigation(ev)
print(f"Investigation opened: {investigation.inv_id}")
print(f"  Triggered by  : {investigation.triggered_by}")
print(f"  Event severity: {ev.severity}  confidence: {ev.confidence}")
print(f"  Tags          : {ev.tags}")
store.print_audit_trail()
print("\nUnified data model ready. All agents will read from and write to this store.")


## Demo 2: Specialist Worker Agents — Four Domain Experts in Parallel

**The Architecture:** Each chapter (70, 72, 75, 83) becomes a specialist worker agent.
They analyze the SAME event independently, from their own domain perspective.

| Worker | Chapter | What It Sees |
|--------|---------|-------------|
| `ThreatDetectionWorker` | Ch70 | Behavioral anomalies, UEBA baseline violations |
| `SIEMCorrelationWorker` | Ch72 | Log correlation, multi-stage attack patterns |
| `AnomalyWorker` | Ch75 | NetFlow reconstruction error, EWMA spikes |
| `ComplianceWorker` | Ch83 | Config drift, policy violations, control failures |

Each worker:
- Has a narrow, specific role
- Uses its own LLM system prompt tuned to its domain
- Returns a structured `WorkerFinding` with confidence score
- Knows its limits: "not in my domain" is a valid answer

> **Network Analogy:** This is like having four NOC specialists look at the same incident.
> The routing engineer, the security engineer, the compliance officer, and the monitoring analyst
> each see different aspects. The coordinator (coming next) synthesizes all four views.


In [ ]:
# Demo 2: Specialist Worker Agents

class ThreatDetectionWorker:
    """Worker 1: UEBA behavioral anomaly analysis (Chapter 70 logic)."""

    SYSTEM = (
        "You are a threat detection specialist. Analyze authentication and behavioral events "
        "for anomalies. Check: impossible travel, new device, odd hours, velocity, geo mismatch, "
        "MFA bypass, privilege escalation. "
        "Return JSON: {finding, confidence, mitre_technique, severity, indicators: [...]}"
    )

    def analyze(self, event: SecurityEvent) -> WorkerFinding:
        """Analyze event for behavioral anomalies."""
        prompt = (
            f"Analyze this authentication event for behavioral anomalies (UEBA):\n"
            f"user={event.user}, src_ip={event.src_ip}, dst={event.device}, "
            f"geo={event.raw_data.get('geo_country')}, normal_geo={event.raw_data.get('normal_geo')}, "
            f"last_login={event.raw_data.get('last_login_hours_ago')}h ago, "
            f"mfa_used={event.raw_data.get('mfa_used')}, tags={event.tags}"
        )
        raw = llm_query(prompt, system=self.SYSTEM, max_tokens=300)
        try:
            data = json.loads(raw)
            conf = float(data.get("confidence", 0.5))
            sev  = data.get("severity", "MEDIUM")
            finding = data.get("finding", raw[:120])
            mitre = [data.get("mitre_technique", "")] if data.get("mitre_technique") else []
        except (json.JSONDecodeError, ValueError):
            conf, sev, finding, mitre = 0.5, "MEDIUM", raw[:120], []
        return WorkerFinding("threat_detection", event.event_id, finding, conf, sev, mitre, raw)


class SIEMCorrelationWorker:
    """Worker 2: Log correlation and multi-stage attack pattern detection (Chapter 72 logic)."""

    SYSTEM = (
        "You are a SIEM correlation specialist. Analyze security events for multi-stage attack patterns. "
        "Cross-reference: brute force precursors, credential reuse, lateral movement, data staging. "
        "Return JSON: {pattern_detected, related_events: [...], attack_stage, confidence, mitre_chain: [...]}"
    )

    def analyze(self, event: SecurityEvent) -> WorkerFinding:
        """Correlate event against known multi-stage attack patterns."""
        prompt = (
            f"Correlate this security event with known attack patterns (SIEM analysis):\n"
            f"type={event.source_type}, user={event.user}, src_ip={event.src_ip}, "
            f"dst_device={event.device}, dst_port={event.raw_data.get('dst_port')}, "
            f"bytes_transferred={event.raw_data.get('bytes_transferred', 0):,}, "
            f"auth_result={event.raw_data.get('auth_result')}"
        )
        raw = llm_query(prompt, system=self.SYSTEM, max_tokens=300)
        try:
            data = json.loads(raw)
            conf = float(data.get("confidence", 0.5))
            sev  = "CRITICAL" if "exfil" in data.get("pattern_detected","").lower() else "HIGH"
            finding = data.get("pattern_detected", raw[:120])
            mitre = data.get("mitre_chain", [])
        except (json.JSONDecodeError, ValueError):
            conf, sev, finding, mitre = 0.5, "HIGH", raw[:120], []
        return WorkerFinding("siem_correlation", event.event_id, finding, conf, sev, mitre, raw)


class AnomalyWorker:
    """Worker 3: NetFlow reconstruction error and traffic anomaly detection (Chapter 75 logic)."""

    SYSTEM = (
        "You are a network anomaly detection specialist. Analyze NetFlow and traffic patterns "
        "for statistical anomalies. Focus on: reconstruction error above threshold, EWMA spikes, "
        "volume anomalies, exfiltration patterns, beaconing. "
        "Return JSON: {anomaly, ewma_spike, confidence, severity, detail}"
    )

    def analyze(self, event: SecurityEvent) -> WorkerFinding:
        """Analyze NetFlow characteristics for anomalies."""
        bytes_gb = event.raw_data.get("bytes_transferred", 0) / 1e9
        prompt = (
            f"Analyze NetFlow and traffic patterns for anomalies:\n"
            f"src_ip={event.src_ip} (external/untrusted), dst={event.device}, "
            f"bytes_transferred={bytes_gb:.2f} GB, duration_minutes=8, "
            f"direction=outbound, dst_port={event.raw_data.get('dst_port')}, "
            f"geo_country={event.raw_data.get('geo_country')}"
        )
        raw = llm_query(prompt, system=self.SYSTEM, max_tokens=250)
        try:
            data = json.loads(raw)
            conf = float(data.get("confidence", 0.5))
            sev  = data.get("severity", "HIGH")
            finding = data.get("anomaly", raw[:120])
        except (json.JSONDecodeError, ValueError):
            conf, sev, finding = 0.5, "HIGH", raw[:120]
        return WorkerFinding("anomaly_detection", event.event_id, finding, conf, sev, [], raw)


class ComplianceWorker:
    """Worker 4: Compliance drift and control failure detection (Chapter 83 logic)."""

    SYSTEM = (
        "You are a compliance monitoring specialist. Analyze security events for compliance "
        "violations: MFA bypass (PCI-8.3.1), logging gaps (SOC2-CC7.2), access control "
        "failures (HIPAA-164.312), config drift. "
        "Return JSON: {drift_detected, control_failures: [...], severity, evidence_logged}"
    )

    def analyze(self, event: SecurityEvent) -> WorkerFinding:
        """Check if the event indicates compliance violations."""
        prompt = (
            f"Analyze this security event for compliance violations:\n"
            f"mfa_used={event.raw_data.get('mfa_used')}, "
            f"user={event.user}, accessing={event.device}, "
            f"src_ip_type=external-tor-exit-node, "
            f"config_check: aaa fallback to local enabled on {event.device}"
        )
        raw = llm_query(prompt, system=self.SYSTEM, max_tokens=250)
        try:
            data = json.loads(raw)
            failures = data.get("control_failures", [])
            conf = 0.92 if failures else 0.6
            sev  = "HIGH" if failures else "LOW"
            finding = f"{len(failures)} control failures: {', '.join(failures[:2])}" if failures else "No violations"
        except (json.JSONDecodeError, ValueError):
            conf, sev, finding = 0.5, "MEDIUM", raw[:120]
        return WorkerFinding("compliance", event.event_id, finding, conf, sev, [], raw)


# ── Run all four workers against the same event ───────────────────────────────
print("Dispatching four specialist workers...\n")
workers = [ThreatDetectionWorker(), SIEMCorrelationWorker(), AnomalyWorker(), ComplianceWorker()]
findings = []
for worker in workers:
    f = worker.analyze(ev)
    findings.append(f)
    investigation.worker_findings.append(f)
    print(f"[{f.worker_name.upper():<22}] sev={f.severity:<8} conf={f.confidence:.2f}  {f.finding[:80]}")

store.update_investigation(investigation)
print(f"\nAll 4 workers complete. Investigation {investigation.inv_id} has {len(findings)} findings.")
print("Average confidence:", round(sum(f.confidence for f in findings) / len(findings), 2))


## Demo 3: Security Coordinator — Synthesis & Incident Brief

**The Problem:** Four workers returned four different findings in four different formats.
Each is valuable alone. But an analyst cannot look at four separate reports simultaneously
and form a coherent picture — especially when handling 200+ events per shift.

**The Solution:** The **Security Coordinator** receives all worker findings and synthesizes
them into a single **Incident Brief** — the only document the analyst needs to read.

**What the coordinator does:**
1. Collects all worker findings for an investigation
2. Detects contradictions (worker A says CRITICAL, worker B says LOW) and explains why
3. Builds the **attack chain** (what happened, in what order, per MITRE ATT&CK)
4. Identifies **affected assets** beyond the initial event
5. Determines overall severity and confidence (weighted by worker confidence scores)
6. Drafts **recommended response actions**
7. Decides: can any actions auto-execute, or does this need human approval?

> **In Simple Words:** Think of the coordinator as the senior analyst who reads all the
> junior analyst reports and writes the executive summary. Except it takes 2 seconds
> instead of 45 minutes — and it never misses a detail.


In [ ]:
# Demo 3: Security Coordinator

COORDINATOR_SYSTEM = (
    "You are a senior security operations coordinator. "
    "You receive findings from multiple specialist AI workers and synthesize them "
    "into a unified incident brief for human analysts. "
    "Be precise about severity, confidence, MITRE techniques, and response actions. "
    "Always state whether human approval is required."
)

def build_coordinator_prompt(event: SecurityEvent,
                              findings: List[WorkerFinding]) -> str:
    """Build the synthesis prompt for the coordinator."""
    findings_text = ""
    for f in findings:
        findings_text += (
            f"\n[{f.worker_name.upper()}]"
            f"  severity={f.severity}  confidence={f.confidence:.2f}"
            f"\n  finding: {f.finding}"
            f"\n  mitre: {f.mitre_tags if f.mitre_tags else 'N/A'}\n"
        )

    return (
        f"SECURITY INCIDENT SYNTHESIS REQUEST\n"
        f"{'='*50}\n"
        f"TRIGGERING EVENT:\n"
        f"  event_id  : {event.event_id}\n"
        f"  user      : {event.user}\n"
        f"  src_ip    : {event.src_ip} (geo: {event.raw_data.get('geo_country')} | normal: {event.raw_data.get('normal_geo')})\n"
        f"  target    : {event.device}\n"
        f"  timestamp : {event.timestamp}\n"
        f"  tags      : {event.tags}\n"
        f"\nSPECIALIST WORKER FINDINGS:\n{findings_text}"
        f"\nSynthesize the above into a unified incident brief with:\n"
        f"1. OVERALL SEVERITY (CRITICAL/HIGH/MEDIUM/LOW) and CONFIDENCE (0.0-1.0)\n"
        f"2. ATTACK CHAIN (ordered steps, MITRE technique per step)\n"
        f"3. AFFECTED ASSETS (primary + secondary blast radius)\n"
        f"4. RECOMMENDED ACTIONS (sorted by priority: IMMEDIATE / WITHIN_1H / WITHIN_24H)\n"
        f"5. HUMAN APPROVAL REQUIRED? (yes if CRITICAL severity OR confidence < 0.85)\n"
        f"\nBe concise. This brief goes directly to an on-call analyst."
    )

def run_coordinator(event: SecurityEvent,
                    findings: List[WorkerFinding],
                    investigation: Investigation) -> Investigation:
    """Coordinator synthesizes all findings into a unified investigation record."""
    prompt = build_coordinator_prompt(event, findings)
    brief  = llm_query(prompt, system=COORDINATOR_SYSTEM,
                       max_tokens=500, model="claude-sonnet-4-6")

    # Parse severity and confidence from the brief
    sev_map = {"CRITICAL": 4, "HIGH": 3, "MEDIUM": 2, "LOW": 1}
    brief_upper = brief.upper()
    if "CRITICAL" in brief_upper:
        investigation.severity = "CRITICAL"
    elif "HIGH" in brief_upper:
        investigation.severity = "HIGH"
    else:
        investigation.severity = "MEDIUM"

    # Weighted average confidence from workers
    investigation.confidence = round(
        sum(f.confidence for f in findings) / len(findings), 2
    )

    # Extract attack chain from findings (simplified)
    all_mitre = []
    for f in findings:
        all_mitre.extend(f.mitre_tags)
    investigation.attack_chain = list(dict.fromkeys(all_mitre))  # deduplicate, preserve order

    # Affected assets
    investigation.affected_assets = [
        event.device,
        event.user + " account",
        "customer PII database",
        "downstream services via db-server-prod-01",
    ]

    investigation.incident_brief = brief
    investigation.status = "ESCALATED"
    return investigation

# ── Run the coordinator ───────────────────────────────────────────────────────
print("Running Security Coordinator synthesis...\n")
investigation = run_coordinator(ev, investigation.worker_findings, investigation)
store.update_investigation(investigation)

print(f"{'='*60}")
print(f"  INCIDENT BRIEF -- {investigation.inv_id}")
print(f"  Severity  : {investigation.severity}")
print(f"  Confidence: {investigation.confidence}")
print(f"  MITRE     : {' -> '.join(investigation.attack_chain) if investigation.attack_chain else 'See brief'}")
print(f"  Assets    : {investigation.affected_assets[:2]} + {len(investigation.affected_assets)-2} more")
print(f"{'='*60}")
print(investigation.incident_brief[:800])
print(f"\nCoordinator synthesis complete. Status: {investigation.status}")


## Demo 4: Actor-Critic Response Chain — Safe Automated Response

**The Problem:** The coordinator says "ISOLATE db-server-prod-01". But:
- Is that host in the critical infrastructure list?
- Will isolation break 5 downstream services?
- Is the action reversible?
- Is the confidence high enough to act without human review?

Blindly executing AI-recommended actions in production is how you create the most
expensive false positive of the year.

**The Solution:** The **Actor-Critic** pattern adds a safety layer:

```
Coordinator -> Actor proposes action -> Critic validates safety -> Decision
                                              |
                               APPROVED_AUTO -> execute immediately
                               APPROVED_CONDITIONS -> execute after conditions met
                               REJECTED / CRITICAL -> escalate to human queue
```

The Actor is optimized for *detection sensitivity* (act fast, catch everything).
The Critic is optimized for *safety* (don't break production, minimize blast radius).

> **Network Analogy:** This is like a BGP change during a production incident.
> You (actor) want to withdraw the prefix NOW. Your colleague (critic) checks:
> "Will this black-hole traffic to 5 downstream sites? Is there a return path?"
> The change only happens after the critic's review. Same pattern, AI agents.

**Walk-Phase Policy (what we implement here):**
- LOW risk, reversible -> auto-execute
- MEDIUM/HIGH risk, production asset -> human approval required
- CRITICAL assets -> always human approval, no exceptions


In [ ]:
# Demo 4: Actor-Critic Response Chain

# Response playbook: action_type -> (base_risk, reversible, auto_eligible)
RESPONSE_PLAYBOOK = {
    "ADD_WATCHLIST":   ("LOW",    True,  True),   # just monitoring - always safe
    "NOTIFY_ONCALL":   ("LOW",    True,  True),   # notification only
    "ENRICH_TICKET":   ("LOW",    True,  True),   # ticket update only
    "BLOCK_IP":        ("MEDIUM", True,  False),  # may affect legitimate traffic
    "DISABLE_ACCOUNT": ("HIGH",   True,  False),  # user impact
    "ISOLATE_HOST":    ("HIGH",   True,  False),  # service disruption risk
    "REVOKE_CERT":     ("HIGH",   False, False),  # hard to reverse
    "WIPE_ENDPOINT":   ("CRITICAL", False, False), # irreversible
}

# Critical infrastructure list -- never auto-act on these
CRITICAL_ASSETS = {"db-server-prod-01", "core-router-01", "auth-server-01", "payment-gw-01"}


class ActionActor:
    """Proposes response actions based on investigation findings."""

    def propose_actions(self, inv: Investigation) -> List[ResponseAction]:
        """Generate a prioritized list of response actions from the incident brief."""
        actions = []

        # Always: notify and enrich ticket (low risk)
        actions.append(ResponseAction(
            action_id  = str(uuid.uuid4())[:8],
            inv_id     = inv.inv_id,
            action_type = "NOTIFY_ONCALL",
            target     = "soc-team@company.com",
            parameters = {"severity": inv.severity, "brief_summary": inv.incident_brief[:200]},
        ))

        # Add compromised IP to watchlist (low risk)
        actions.append(ResponseAction(
            action_id  = str(uuid.uuid4())[:8],
            inv_id     = inv.inv_id,
            action_type = "ADD_WATCHLIST",
            target     = "185.220.101.47",
            parameters = {"reason": "Tor exit node - confirmed attack source", "ttl_hours": 168},
        ))

        # If CRITICAL: propose account disable and host isolation
        if inv.severity in ("CRITICAL", "HIGH"):
            actions.append(ResponseAction(
                action_id  = str(uuid.uuid4())[:8],
                inv_id     = inv.inv_id,
                action_type = "DISABLE_ACCOUNT",
                target     = "admin.chen",
                parameters = {"reason": "Credential compromise - impossible travel", "notify_user": True},
            ))
            actions.append(ResponseAction(
                action_id  = str(uuid.uuid4())[:8],
                inv_id     = inv.inv_id,
                action_type = "ISOLATE_HOST",
                target     = "db-server-prod-01",
                parameters = {"vlan": "incident-response", "rollback_hours": 4, "allow_ir_team": True},
            ))

        return actions


class ActionCritic:
    """Validates proposed actions against safety policy before execution."""

    CRITIC_SYSTEM = (
        "You are a security response safety critic. "
        "Evaluate proposed security response actions for: reversibility, blast radius, "
        "asset criticality, confidence threshold (must be >= 0.85 for auto-execution), "
        "and compliance with Walk-phase policy (no auto-action on production assets). "
        "Return JSON: {verdict, risk_level, reversible, blast_radius, conditions: [...], "
        "auto_execute, reason}"
    )

    def evaluate(self, action: ResponseAction,
                 inv: Investigation) -> ResponseAction:
        """Critic evaluates and annotates the proposed action."""
        base_risk, reversible, auto_eligible = RESPONSE_PLAYBOOK.get(
            action.action_type, ("HIGH", False, False)
        )

        # Hard rules: critical assets NEVER auto-execute
        is_critical_asset = action.target in CRITICAL_ASSETS
        conf_ok = inv.confidence >= 0.85

        prompt = (
            f"Evaluate this proposed security response action:\n"
            f"  action_type  : {action.action_type}\n"
            f"  target       : {action.target}\n"
            f"  parameters   : {action.parameters}\n"
            f"  investigation: severity={inv.severity}, confidence={inv.confidence:.2f}\n"
            f"  base_risk    : {base_risk}\n"
            f"  in_critical_assets: {is_critical_asset}\n"
            f"  confidence_ok: {conf_ok} (threshold 0.85)\n"
            f"  walk_phase_policy: no auto-action on HIGH/CRITICAL risk or production assets\n"
            f"\nShould this action auto-execute, require conditions, or need human approval?"
        )

        raw = llm_query(prompt, system=self.CRITIC_SYSTEM, max_tokens=300)
        try:
            data = json.loads(raw)
            action.critic_verdict = data.get("verdict", "REJECTED")
            action.auto_execute   = data.get("auto_execute", False) and not is_critical_asset
            action.rollback_notes = str(data.get("conditions", []))
        except (json.JSONDecodeError, ValueError):
            # Safe default: require human approval when we cannot parse critic output
            action.critic_verdict = "ESCALATE_TO_HUMAN"
            action.auto_execute   = False

        # Override: critical assets always go to human queue
        if is_critical_asset:
            action.critic_verdict = "REQUIRE_HUMAN_APPROVAL"
            action.auto_execute   = False
            action.rollback_notes += " [CRITICAL ASSET -- human approval mandatory]"

        return action

    def execute_or_escalate(self, action: ResponseAction) -> str:
        """Execute approved low-risk actions; escalate others."""
        if action.auto_execute and action.critic_verdict == "APPROVED":
            action.approved_by = "autonomous"
            action.executed_at = datetime.datetime.now().isoformat(timespec="seconds")
            return f"AUTO-EXECUTED  [{action.action_type}] -> {action.target}"
        else:
            return f"ESCALATED      [{action.action_type}] -> {action.target}  (verdict: {action.critic_verdict})"


# ── Run Actor-Critic chain ────────────────────────────────────────────────────
actor  = ActionActor()
critic = ActionCritic()

proposed = actor.propose_actions(investigation)
print(f"Actor proposed {len(proposed)} response actions:\n")

for action in proposed:
    action = critic.evaluate(action, investigation)
    result = critic.execute_or_escalate(action)
    print(f"  {result}")
    print(f"    verdict={action.critic_verdict}  auto={action.auto_execute}  reversible={action.reversible}")
    if action.rollback_notes:
        notes = action.rollback_notes[:80]
        print(f"    notes  : {notes}")

print(f"\nActor-Critic chain complete. Low-risk actions executed. High-risk escalated to human queue.")


## Demo 5: Full AI Security Fabric — End-to-End Pipeline

**This is it.** Everything from Chapters 70-83 wired into a single pipeline.

```
Raw Events (any source)
        |
        v
[SecurityEventStore]  -- unified schema (Demo 1)
        |
        v
[Four Worker Agents]  -- parallel specialist analysis (Demo 2)
        |
        v
[Security Coordinator] -- synthesis + incident brief (Demo 3)
        |
        v
[Actor-Critic Chain]  -- propose + validate + execute/escalate (Demo 4)
        |
        v
[Immutable Audit Log] -- every decision logged with SHA-256 chain
        |
        v
[Analyst Dashboard]   -- prioritized Investigation queue
                         NOT 50,000 raw alerts. Just 30 investigated incidents.
```

**The result:** An analyst receives a pre-built incident brief with:
- What happened (attack chain, MITRE mapping)
- What is affected (primary + blast radius assets)
- What was done automatically (low-risk actions already executed)
- What requires their decision (high-risk actions in approval queue)
- Full audit trail for compliance

**MTTD before AI:** 197 days industry average (Ponemon Institute)
**MTTD with AI Security Fabric:** Under 1 hour for pattern-matched attacks


In [ ]:
# Demo 5: Full AI Security Fabric

class AISecurityFabric:
    """
    End-to-end AI Security Fabric.
    Integrates: unified data model + specialist workers + coordinator +
    actor-critic response + immutable audit trail + analyst dashboard.
    """

    def __init__(self):
        self._store      = SecurityEventStore()
        self._workers    = [
            ThreatDetectionWorker(),
            SIEMCorrelationWorker(),
            AnomalyWorker(),
            ComplianceWorker(),
        ]
        self._actor  = ActionActor()
        self._critic = ActionCritic()
        self._metrics = {
            "events_ingested": 0,
            "investigations_opened": 0,
            "actions_auto_executed": 0,
            "actions_escalated": 0,
            "mttd_seconds": [],
        }

    def ingest_event(self, raw: Dict, source_type: str = "auth_log") -> SecurityEvent:
        """Ingest a raw event and normalize it into the unified schema."""
        ev = SecurityEvent(
            event_id    = raw.get("id", str(uuid.uuid4())[:8]),
            timestamp   = raw.get("timestamp", datetime.datetime.now().isoformat()),
            source_type = source_type,
            device      = raw.get("dst_device", "unknown"),
            src_ip      = raw.get("src_ip", "0.0.0.0"),
            dst_ip      = raw.get("dst_ip", "unknown"),
            user        = raw.get("user", "unknown"),
            action      = "authentication_success" if raw.get("auth_result") == "success" else "event",
            confidence  = 0.90,
            severity    = "HIGH",
            raw_data    = raw,
            tags        = raw.get("tags", []),
        )
        self._store.ingest(ev)
        self._metrics["events_ingested"] += 1
        return ev

    def _detect_and_triage(self, event: SecurityEvent) -> bool:
        """Quick triage: should this event open a full investigation?"""
        # Open investigation for HIGH/CRITICAL confidence events
        # In production: use ML triage model here
        high_risk_tags = {"tor-exit-node", "geo-mismatch", "no-mfa", "prod-db"}
        return bool(set(event.tags) & high_risk_tags) or event.severity in ("HIGH", "CRITICAL")

    def investigate(self, event: SecurityEvent) -> Investigation:
        """Full investigation pipeline: workers -> coordinator -> actor-critic."""
        t_start = time.time()

        # Open investigation
        inv = self._store.open_investigation(event)
        self._metrics["investigations_opened"] += 1

        # Dispatch all workers (production: asyncio.gather for true parallelism)
        print(f"  [Workers] Dispatching {len(self._workers)} specialists...")
        for worker in self._workers:
            finding = worker.analyze(event)
            inv.worker_findings.append(finding)

        # Coordinator synthesis
        print(f"  [Coordinator] Synthesizing {len(inv.worker_findings)} findings...")
        inv = run_coordinator(event, inv.worker_findings, inv)
        self._store.update_investigation(inv)

        # Actor-Critic response chain
        print(f"  [Actor-Critic] Evaluating response actions...")
        proposed = self._actor.propose_actions(inv)
        for action in proposed:
            action = self._critic.evaluate(action, inv)
            result = self._critic.execute_or_escalate(action)
            if action.auto_execute:
                self._metrics["actions_auto_executed"] += 1
            else:
                self._metrics["actions_escalated"] += 1

        # Record MTTD
        mttd = round(time.time() - t_start, 2)
        self._metrics["mttd_seconds"].append(mttd)

        return inv

    def process(self, raw_events: List[Dict]) -> List[Investigation]:
        """Process a batch of raw events through the full fabric."""
        investigations = []
        print(f"\n{'='*60}")
        print(f"  AI SECURITY FABRIC -- Processing {len(raw_events)} events")
        print(f"{'='*60}\n")

        for raw in raw_events:
            event = self.ingest_event(raw)
            if self._detect_and_triage(event):
                print(f"HIGH-CONFIDENCE EVENT: {event.event_id} ({event.tags[0] if event.tags else 'no-tags'})")
                inv = self.investigate(event)
                investigations.append(inv)
            else:
                print(f"LOW-RISK EVENT: {event.event_id} -> buffered for batch analysis")

        return investigations

    def analyst_dashboard(self, investigations: List[Investigation]):
        """Print the analyst dashboard -- the single pane of glass."""
        SEV_ORDER = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3, "UNKNOWN": 4}
        sorted_inv = sorted(investigations, key=lambda i: SEV_ORDER.get(i.severity, 4))

        print(f"\n{'='*60}")
        print(f"  SOC ANALYST DASHBOARD -- {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}")
        print(f"  Events ingested  : {self._metrics['events_ingested']}")
        print(f"  Investigations   : {len(investigations)}")
        print(f"  Auto-executed    : {self._metrics['actions_auto_executed']} actions")
        print(f"  Pending approval : {self._metrics['actions_escalated']} actions")
        avg_mttd = (sum(self._metrics['mttd_seconds']) / len(self._metrics['mttd_seconds'])
                    if self._metrics['mttd_seconds'] else 0)
        print(f"  Avg MTTD        : {avg_mttd:.1f}s (AI pipeline)")
        print(f"{'='*60}")

        print(f"\n  INVESTIGATION QUEUE (sorted by severity):")
        for inv in sorted_inv:
            print(f"\n  [{inv.severity:<8}] {inv.inv_id}  conf={inv.confidence:.2f}  status={inv.status}")
            print(f"  Assets : {', '.join(inv.affected_assets[:2])}")
            chain_str = " -> ".join(inv.attack_chain[:3]) if inv.attack_chain else "See brief"
            print(f"  Chain  : {chain_str}")
            print(f"  Brief  :\n  {inv.incident_brief[:300].replace(chr(10), chr(10)+'  ')}")

        self._store.print_audit_trail(last_n=8)

# ── Run the full AI Security Fabric ──────────────────────────────────────────
# Feed it the suspicious event plus two benign events to show triage
EVENT_BATCH = [
    # The main case study event (CRITICAL)
    {**SUSPICIOUS_EVENT, "tags": ["tor-exit-node", "geo-mismatch", "no-mfa", "prod-db"]},
    # A routine event (LOW -- should NOT open investigation)
    {"id": "EVT-2026-0043", "timestamp": "2026-02-26T03:15:00Z",
     "user": "bob.smith", "src_ip": "10.0.1.10", "dst_device": "web-server-01",
     "auth_result": "success", "mfa_used": True, "geo_country": "US",
     "normal_geo": "US", "bytes_transferred": 50000, "dst_port": 443,
     "tags": []},
]

fabric = AISecurityFabric()
investigations = fabric.process(EVENT_BATCH)
fabric.analyst_dashboard(investigations)

print(f"\nAI Security Fabric complete.")
print(f"Instead of {50000} raw events, the analyst sees {len(investigations)} investigated incidents.")


## Chapter 87 Summary — The AI Security Fabric

You built the complete **AI Security Fabric** -- the unified architecture that integrates
every chapter of Volume 4 into a production security operations system.

| Demo | Component | Chapter Source | Key Capability |
|------|-----------|---------------|---------------|
| 1 | Unified Data Model | Architecture | SecurityEvent + Investigation + ResponseAction schemas |
| 2 | Specialist Worker Agents | Ch70+72+75+83 | Four parallel domain experts, structured findings |
| 3 | Security Coordinator | Multi-agent theory | Synthesizes workers into incident brief, MITRE mapping |
| 4 | Actor-Critic Response | Safety pattern | Proposes actions, validates safety, auto/escalate |
| 5 | Full AI Security Fabric | Integration | End-to-end pipeline with analyst dashboard |

### The Three Design Principles

**1. Crawl-Walk-Run:** Do not automate responses before you have trust data.
Start with AI-generated briefs only (Crawl). Auto-execute low-risk actions after
85%+ analyst agreement rate (Walk). Full supervised autonomy after 90%+ and zero
catastrophic false positives (Run).

**2. Unified Data Model First:** Every component must read from and write to the
shared schema. Schema drift is the most common cause of the "five isolated systems"
anti-pattern. Treat the schema as a network interface standard -- immutable contract.

**3. Measure MTTD from Day One:** If you cannot measure it before AI, you cannot
prove improvement after. The industry average without AI is 197 days. A well-tuned
AI Security Fabric targets under 1 hour for pattern-matched attacks.

### Production Deployment Path

```
Month 1-2  : Audit and fix data collection (garbage in = garbage out)
Month 3-4  : Deploy unified data model + SIEM worker only (Crawl)
Month 5-6  : Add threat detection + anomaly workers. Build trust calibration data.
Month 7-8  : Deploy coordinator. Measure analyst agreement rate.
Month 9-10 : Enable auto-execution for low-risk actions (Walk)
Month 11-12: Add compliance worker. Full fabric. Measure MTTD improvement.
Month 12+  : Consider Run-phase for stable action categories with proven track record.
```

**This concludes Volume 4: Security Operations.**
Volume 5 will apply the same agent and RAG architectures to Network Automation.


In [ ]:
# Integration Example: Production Deployment Checklist
# This shows what you would add to move from this notebook to real deployment.

PRODUCTION_CHECKLIST = [
    # Data Layer
    ("Data", "Replace SecurityEventStore with Kafka + ClickHouse for streaming ingestion"),
    ("Data", "Implement LSM-Tree storage (RocksDB/ScyllaDB) for write performance"),
    ("Data", "Add CIM schema validation middleware -- reject non-conforming events"),

    # Worker Layer
    ("Workers", "Use asyncio.gather() to dispatch all workers truly in parallel"),
    ("Workers", "Add circuit breaker: if worker fails 3x, coordinator skips it"),
    ("Workers", "Connect workers to real data: NetFlow via IPFIX, auth logs via SIEM API"),

    # Coordinator Layer
    ("Coordinator", "Add conflict resolution logic: log contradictions, escalate to human"),
    ("Coordinator", "Implement confidence decay: reduce conf if workers disagree >40%"),
    ("Coordinator", "Store incident briefs in PostgreSQL with full-text search"),

    # Response Layer
    ("Response", "Wire BLOCK_IP to firewall API (Palo Alto / Cisco FMC)"),
    ("Response", "Wire ISOLATE_HOST to SDN controller or NSX microsegmentation API"),
    ("Response", "Wire DISABLE_ACCOUNT to Active Directory / Okta via SCIM"),
    ("Response", "Add rollback scheduler: auto-restore after N hours if no analyst action"),

    # Observability
    ("Metrics",  "Export MTTD, MTTR, CFR, alert-to-incident ratio to Grafana"),
    ("Metrics",  "Alert on detection category distribution drift (feedback loop indicator)"),
    ("Metrics",  "Track analyst agreement rate per action category for Walk->Run gating"),

    # Security
    ("Security", "Route all LLM prompts through Ch80 guardrail stack before sending"),
    ("Security", "Write every Investigation + ResponseAction to Ch83 immutable evidence log"),
    ("Security", "Run red team exercises quarterly to test for feedback loop blind spots"),
]

print("Production Deployment Checklist -- AI Security Fabric")
print("="*60)
current_section = ""
for (section, item) in PRODUCTION_CHECKLIST:
    if section != current_section:
        print(f"\n  [{section.upper()}]")
        current_section = section
    print(f"    - {item}")

print(f"\n{'='*60}")
print(f"\nChapter 87 -- Complete Security Case Study: DONE!")
print(f"Volume 4: Security Operations: COMPLETE!")
print()
print("Components of the AI Security Fabric:")
fabric_components = [
    ("SecurityEventStore",   "Unified schema -- shared contract for all agents"),
    ("ThreatDetectionWorker","Ch70 UEBA behavioral anomaly analysis"),
    ("SIEMCorrelationWorker","Ch72 log correlation and attack pattern detection"),
    ("AnomalyWorker",        "Ch75 NetFlow reconstruction error and EWMA"),
    ("ComplianceWorker",     "Ch83 policy-as-code and drift detection"),
    ("SecurityCoordinator",  "Synthesizes 4 workers into incident brief"),
    ("ActionActor",          "Proposes prioritized response actions"),
    ("ActionCritic",         "Validates safety: auto-execute or escalate"),
    ("AISecurityFabric",     "End-to-end pipeline with analyst dashboard"),
]
for name, desc in fabric_components:
    print(f"  {name:<28} {desc}")

print()
print("Next: Volume 5 -- AI for Network Automation")
